# 워커 CPU 스레드 수 비교 — GPU 서버 실행 (ai_2_to_4 3-C)

11/10 측정에서 곡을 처리하는 동안 6번 GPU 사용률은 평균 3~22%였고 서버 load는 코어 256개에 170~270이었다. GPU보다 CPU 쪽에서 시간을 쓰는지 보려고 워커의 CPU 스레드 수(OMP·MKL·OpenBLAS·torch)만 바꿔 같은 10곡을 돌린다.

- `worker_speed_server.ipynb`와 같은 방식: fake BE에 실제 발매곡 10곡(mp3)을 넣고 운영과 같은 `ssaplash-worker run`으로 사이트 흐름 전체를 곡마다 끝까지 돌림
- 채보는 새로 반영할 드럼 킷 분리 겹침 2 설정만 씀
- 스레드 수 기본값(제한 없음)·16·4 세 가지. 순서를 기본값 → 16 → 4 → 4 → 16 → 기본값으로 두 번 돌려, 시간이 지나며 서버 상태가 바뀐 효과가 한 설정에만 몰리지 않게 함. 설정마다 20곡
- 곡마다 워커 프로세스 CPU 사용률(코어 하나 = 100)·GPU 사용률·서버 load를 같이 기록

**준비물.** `worker_speed_server.ipynb` 때 올린 `ssaplash/upload/speed_songs/`(mp3 10곡)를 그대로 씀. 이 파일과 로컬 `AI/scripts/worker_speed_check.py`(11/10 오후 수정본)를 JupyterHub `ssaplash/notebooks/`에 올린다(스크립트는 덮어쓰기).

**순서.** 1번 셀을 한 번 실행하면 백그라운드로 끝까지 돈다(40~50분). 2번 셀은 여러 번 쳐도 되고, 끝나면 결과를 출력하고 `ssaplash/results/worker_threads.zip`으로 묶는다.

**끊기면.** 1번은 nohup이라 브라우저를 닫아도 끝까지 돈다. 커널이 다시 시작됐으면 2번만 다시 치면 된다.

In [ ]:
# 1. 설정·점검·측정 시작 (한 번만. 이미 시작했으면 다시 시작하지 않고 알려 줌)
import getpass
import glob
import os
import subprocess
import zipfile

GPU_ID = "6"
HOME = os.path.expanduser("~")
WORK = f"{HOME}/ssaplash"
AI = f"{WORK}/S15P31A307/AI"
SCRIPT = f"{WORK}/notebooks/worker_speed_check.py"
SONG_ZIP = f"{WORK}/upload/speed_songs.zip"
SONG_DIR = f"{WORK}/upload/speed_songs"
WORKER_LOG = f"{WORK}/results/worker.log"
OUT = f"{WORK}/results/worker-threads"
LOG = f"{WORK}/results/worker_threads.log"

# 운영 워커와 같은 환경으로 돌림: GPU 6번만 보이게 하고 스레드 수 제한은 걸지 않음 (worker.env와 같음)
# MPLBACKEND: Jupyter 커널의 inline 그림 설정이 넘어가면 워커가 모델을 못 올림. 운영 워커처럼 Agg
env = dict(os.environ, CUDA_VISIBLE_DEVICES=GPU_ID, PYTHONUTF8="1", PYTHONUNBUFFERED="1", MPLBACKEND="Agg")
for k in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "SSAPLASH_WORKER_KEEP_FILES"):
    env.pop(k, None)

!nvidia-smi -i $GPU_ID --query-gpu=index,name,memory.used,memory.total,utilization.gpu --format=csv
!cat /proc/loadavg
wpid = open(f"{WORK}/worker.pid").read().strip() if os.path.isfile(f"{WORK}/worker.pid") else ""
print("운영 워커:", subprocess.run(["ps", "-o", "pid,etime,args", "-p", wpid], capture_output=True, text=True).stdout[-150:] if wpid else "pid 파일 없음")
!tail -1 $WORKER_LOG | cut -c1-150
!cd $AI && git log --oneline -1

assert os.path.isfile(SCRIPT), "ssaplash/notebooks/worker_speed_check.py가 없음. 올리고 다시 실행"
assert "--threads" in open(SCRIPT, encoding="utf-8").read(), "worker_speed_check.py가 옛 버전임. 새로 올려 덮어쓰고 다시 실행"
if not os.path.isdir(SONG_DIR) and os.path.isfile(SONG_ZIP):
    zipfile.ZipFile(SONG_ZIP).extractall(f"{WORK}/upload")
songs = sorted(glob.glob(f"{SONG_DIR}/*.mp3"))
print("측정할 곡", len(songs), "개:", [os.path.basename(s) for s in songs])
assert songs, "곡이 없음. ssaplash/upload/speed_songs.zip을 올리고 다시 실행"

# 전에 한 측정이 곡을 하나도 못 끝냈으면 그 결과 폴더를 옆으로 치우고 새로 시작함
# (summary.md에는 운영 워커 실제 작업의 DONE이 있어서 측정 곡 결과인 runs.csv로 판단)
if os.path.isfile(f"{OUT}/summary.md") and ",DONE," not in open(f"{OUT}/runs.csv", encoding="utf-8").read():
    import time
    os.rename(OUT, f"{OUT}-fail-{time.strftime('%H%M%S')}")
    print("전에 실패한 측정 결과를 치움")
running = subprocess.run(["pgrep", "-u", getpass.getuser(), "-f", "worker_speed_check.py"],
                         capture_output=True, text=True).stdout.split()
if os.path.isfile(f"{OUT}/summary.md"):
    print("이미 끝남. 2번 셀로 결과를 봄 (다시 측정하려면 results/worker-threads 폴더 이름을 바꾸고 이 셀을 다시 실행)")
elif running:
    print("이미 도는 중. pid", running, "→ 2번 셀로 진행을 봄")
else:
    cmd = [f"{AI}/.venv/bin/python", SCRIPT, "--song-dir", SONG_DIR, "--gpu-index", GPU_ID,
           "--worker-log", WORKER_LOG, "--out", OUT,
           "--only-variant", "--threads", "default,16,4", "--rounds", "2"]
    proc = subprocess.Popen(["nohup", *cmd], cwd=AI, stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
                            env=env, start_new_session=True)
    print("시작. pid", proc.pid, "로그", LOG)
# 확인:
#  - GPU 한 줄(H200). memory.used에 운영 워커 몫 5,000MiB 안팎이 잡혀 있는 게 정상.
#    utilization.gpu가 50%를 넘으면 다른 사람이 6번 GPU를 쓰는 중이니 그 숫자를 적어 둠 (측정은 그대로 진행)
#  - loadavg 첫 숫자를 적어 둠 (서버 CPU가 얼마나 바쁜지)
#  - '운영 워커:' 아래 'ssaplash-worker run' 한 줄, 워커 로그 마지막 줄
#  - '측정할 곡 10 개:' 뒤에 allright.mp3 … wenot.mp3
#  - 'worker_speed_check.py가 옛 버전임'이 나오면 스크립트를 다시 올려 덮어쓰고 이 셀을 다시 실행
#  - 마지막 줄 '시작. pid 숫자'. 바로 2번으로

In [ ]:
# 2. 진행 확인·결과 (여러 번 쳐도 됨)
!tail -3 $LOG | cut -c1-200
!pgrep -af worker_speed_check.py | cut -c1-100 || echo "측정 프로세스 없음"
if os.path.isfile(f"{OUT}/summary.md"):
    print(open(f"{OUT}/summary.md", encoding="utf-8").read())
    zpath = f"{WORK}/results/worker_threads.zip"
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(glob.glob(f"{OUT}/*.md") + glob.glob(f"{OUT}/*.csv") + glob.glob(f"{OUT}/*.log")):
            z.write(f, f"worker-threads/{os.path.basename(f)}")
        z.write(LOG, "worker-threads/worker_threads.log")
    print("묶음", zpath, os.path.getsize(zpath) // 1024, "KB")
# 확인:
#  - 도는 중이면 로그 끝에 '[speed] 시각 drumsep_overlap_2_threads_default_r1: 3/10 끝남'처럼 끝난 곡 수가 올라감.
#    threads_default_r1 → threads_16_r1 → threads_4_r1 → threads_4_r2 → threads_16_r2 → threads_default_r2 순서로 6번 돎
#  - 끝나면 '[speed] ... 끝. 저장 .../summary.md' 줄과 표들이 출력되고 '묶음 .../worker_threads.zip 수십 KB' 줄이 찍힘
#  - '측정 프로세스 없음'인데 표가 없으면 실패한 것. 새 셀에서 아래 두 줄을 치고 나온 내용을 그대로 붙여 줌:
#    !tail -20 ~/ssaplash/results/worker_threads.log
#    !tail -20 ~/ssaplash/results/worker-threads/worker_drumsep_overlap_2_threads_default_r1.log